## 1. 기본설정

In [8]:
import shutil # 파일을 복사하기 위한 표준 라이브럴기
from pathlib import Path

import pandas as pd
from IPython.display import display


# 분석할 데이터셋
RATIOS = ["HI", "LI"]
SIZE = "Small"

SAMPLE_ROWS = 10_000

# Kaggle 데이터셋 주소
DATASET_HANDLE = (
    "ealtman2019/"
    "ibm-transactions-for-anti-money-laundering-aml"
)

# 현재 노트북 위치에 datasets 폴더 생성
DATA_DIR = Path.cwd() / "datasets"
DATA_DIR.mkdir(exist_ok=True)

print("현재 작업 폴더:", Path.cwd())
print("데이터 저장 폴더:", DATA_DIR)
print("분석 대상:", RATIOS, SIZE)

현재 작업 폴더: /workspace/money_laundry
데이터 저장 폴더: /workspace/money_laundry/datasets
분석 대상: ['HI', 'LI'] Small


## 2. 다운로드할 파일 목록 만들기

In [12]:
import kagglehub

wanted_files = []

for ratio in RATIOS:
    wanted_files.extend([

        # 전체 거래 데이터
        f"{ratio}-{SIZE}_Trans.csv",

        # 계좌와 엔티티 정보
        f"{ratio}-{SIZE}_accounts.csv",

        # 자금세탁 패턴별 거래 목록
        f"{ratio}-{SIZE}_Patterns.txt"
    ])

print('다운로드 대상 파일:')

for filename in wanted_files:
    print("-", filename)
    

다운로드 대상 파일:
- HI-Small_Trans.csv
- HI-Small_accounts.csv
- HI-Small_Patterns.txt
- LI-Small_Trans.csv
- LI-Small_accounts.csv
- LI-Small_Patterns.txt


## 3. HI_Samll과 LI-Small 파일 다운로드

In [13]:
# 다운로드할 파일을 하나씩 확인
for filename in wanted_files:

    # 최종적으로 파일을 저장할 위치
    destination = DATA_DIR / filename

    # 해당 파일이 이미 datasets 폴더에 있으면 다시 받지 않음
    if destination.exists():
        print("이미 존재하므로 건너뜀:", destination.name)
        continue

    print("다운로드 시작:", filename)

    # Kaggle 캐시 폴더에 파일 하나만 다운로드
    # path=filename을 지정했기 때문에 전체 데이터셋을 받지 않음
    downloaded_path = Path(
        kagglehub.dataset_download(
            DATASET_HANDLE,
            path=filename,
        )
    )

    # Kaggle 캐시에 저장된 파일을 현재 datasets 폴더로 복사
    shutil.copy2(
        downloaded_path,
        destination,
    )

    print("다운로드 완료:", destination)

이미 존재하므로 건너뜀: HI-Small_Trans.csv
이미 존재하므로 건너뜀: HI-Small_accounts.csv
이미 존재하므로 건너뜀: HI-Small_Patterns.txt
이미 존재하므로 건너뜀: LI-Small_Trans.csv
이미 존재하므로 건너뜀: LI-Small_accounts.csv
이미 존재하므로 건너뜀: LI-Small_Patterns.txt


## 4. 다운로드된 파일 확인

In [14]:
# datasets 폴더 안의 파일만 가져와 파일명순으로 정렬
downloaded_files = sorted(
    path
    for path in DATA_DIR.iterdir()
    if path.is_file()
)

print("현재 datasets 폴더의 파일 목록")
print("=" * 70)

for path in downloaded_files:
    # 파일 크기를 byte에서 MB로 변환
    size_mb = path.stat().st_size / (1024 ** 2)

    print(
        f"{path.name:<35}"
        f"{size_mb:>12.2f} MB"
    )

현재 datasets 폴더의 파일 목록
HI-Large_Patterns.txt                     13.17 MB
HI-Large_Trans.csv                     14279.60 MB
HI-Large_accounts.csv                    140.85 MB
HI-Medium_Patterns.txt                     2.17 MB
HI-Medium_Trans.csv                     2891.33 MB
HI-Medium_accounts.csv                   138.29 MB
HI-Small_Patterns.txt                      0.31 MB
HI-Small_Trans.csv                       453.63 MB
HI-Small_accounts.csv                     32.48 MB
LI-Large_Patterns.txt                      1.85 MB
LI-Large_Trans.csv                     15966.91 MB
LI-Large_accounts.csv                    137.66 MB
LI-Medium_Patterns.txt                     0.37 MB
LI-Medium_Trans.csv                     2838.55 MB
LI-Medium_accounts.csv                   135.21 MB
LI-Small_Patterns.txt                      0.09 MB
LI-Small_Trans.csv                       620.29 MB
LI-Small_accounts.csv                     45.06 MB


## 5. HI와 LI 파일 경로를 딕셔너리로 정리

In [15]:
# 데이터셋별 파일 경로를 저장할 딕셔너리
dataset_paths = {}

for ratio in RATIOS:
    dataset_paths[ratio] = {
        "transactions": (
            DATA_DIR / f"{ratio}-{SIZE}_Trans.csv"
        ),
        "accounts": (
            DATA_DIR / f"{ratio}-{SIZE}_accounts.csv"
        ),
        "patterns": (
            DATA_DIR / f"{ratio}-{SIZE}_Patterns.txt"
        ),
    }


# 모든 파일이 실제로 존재하는지 확인
for ratio, paths in dataset_paths.items():
    print(f"\n===== {ratio}-{SIZE} =====")

    for file_type, path in paths.items():
        if not path.exists():
            raise FileNotFoundError(
                f"파일이 존재하지 않습니다: {path}"
            )

        print(f"{file_type:<15}: {path.name}")


===== HI-Small =====
transactions   : HI-Small_Trans.csv
accounts       : HI-Small_accounts.csv
patterns       : HI-Small_Patterns.txt

===== LI-Small =====
transactions   : LI-Small_Trans.csv
accounts       : LI-Small_accounts.csv
patterns       : LI-Small_Patterns.txt


In [16]:
dataset_paths

{'HI': {'transactions': PosixPath('/workspace/money_laundry/datasets/HI-Small_Trans.csv'),
  'accounts': PosixPath('/workspace/money_laundry/datasets/HI-Small_accounts.csv'),
  'patterns': PosixPath('/workspace/money_laundry/datasets/HI-Small_Patterns.txt')},
 'LI': {'transactions': PosixPath('/workspace/money_laundry/datasets/LI-Small_Trans.csv'),
  'accounts': PosixPath('/workspace/money_laundry/datasets/LI-Small_accounts.csv'),
  'patterns': PosixPath('/workspace/money_laundry/datasets/LI-Small_Patterns.txt')}}

## 6. 데이터 로드
- 일부만 할거면 주석 풀기

In [22]:
raw_samples = {}

for ratio in RATIOS:
    print(f"{ratio}-{SIZE} 샘플 데이터 로드 중.. ")

    transactions = pd.read_csv(
        dataset_paths[ratio]['transactions'],
        # nrows = SAMPLE_ROWS,
        low_memory = False,
    )

    accounts = pd.read_csv(
        dataset_paths[ratio]['accounts'],
        # nrows = SAMPLE_ROWS,
        low_memory = False,
    )

    raw_samples[ratio] = {
        "transactions" : transactions,
        "accounts" : accounts,
    }

    print(
        f"{ratio} 거래 데이터크기",
        transactions.shape
    )

    
    print(
        f"{ratio} 계좌 데이터크기",
        accounts.shape
    )



HI-Small 샘플 데이터 로드 중.. 
HI 거래 데이터크기 (5078345, 11)
HI 계좌 데이터크기 (518581, 5)
LI-Small 샘플 데이터 로드 중.. 
LI 거래 데이터크기 (6924049, 11)
LI 계좌 데이터크기 (712688, 5)


## 7. 거래 데이터 확인

In [23]:
hi_trans_raw = raw_samples['HI']["transactions"]

print("===== HI-Small 거래 데이터 =====")
print("현재 불러온 크기:", hi_trans_raw.shape)

hi_trans_raw.info(
    verbose = True,
    show_counts = True,
    memory_usage = "deep",
)

===== HI-Small 거래 데이터 =====
현재 불러온 크기: (5078345, 11)
<class 'pandas.DataFrame'>
RangeIndex: 5078345 entries, 0 to 5078344
Data columns (total 11 columns):
 #   Column              Non-Null Count    Dtype  
---  ------              --------------    -----  
 0   Timestamp           5078345 non-null  str    
 1   From Bank           5078345 non-null  int64  
 2   Account             5078345 non-null  str    
 3   To Bank             5078345 non-null  int64  
 4   Account.1           5078345 non-null  str    
 5   Amount Received     5078345 non-null  float64
 6   Receiving Currency  5078345 non-null  str    
 7   Amount Paid         5078345 non-null  float64
 8   Payment Currency    5078345 non-null  str    
 9   Payment Format      5078345 non-null  str    
 10  Is Laundering       5078345 non-null  int64  
dtypes: float64(2), int64(3), str(6)
memory usage: 699.6 MB


In [26]:
hi_trans_raw.head()

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:20,10,8000EBD30,10,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,0
1,2022/09/01 00:20,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0
2,2022/09/01 00:00,3209,8000F4670,3209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0
3,2022/09/01 00:02,12,8000F5030,12,8000F5030,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,0
4,2022/09/01 00:06,10,8000F5200,10,8000F5200,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,0


In [24]:
li_trans_raw = raw_samples['LI']["transactions"]

print("===== LI-Small 거래 데이터 =====")
print("현재 불러온 크기:", li_trans_raw.shape)

li_trans_raw.info(
    verbose = True,
    show_counts = True,
    memory_usage = "deep",
)

===== LI-Small 거래 데이터 =====
현재 불러온 크기: (6924049, 11)
<class 'pandas.DataFrame'>
RangeIndex: 6924049 entries, 0 to 6924048
Data columns (total 11 columns):
 #   Column              Non-Null Count    Dtype  
---  ------              --------------    -----  
 0   Timestamp           6924049 non-null  str    
 1   From Bank           6924049 non-null  int64  
 2   Account             6924049 non-null  str    
 3   To Bank             6924049 non-null  int64  
 4   Account.1           6924049 non-null  str    
 5   Amount Received     6924049 non-null  float64
 6   Receiving Currency  6924049 non-null  str    
 7   Amount Paid         6924049 non-null  float64
 8   Payment Currency    6924049 non-null  str    
 9   Payment Format      6924049 non-null  str    
 10  Is Laundering       6924049 non-null  int64  
dtypes: float64(2), int64(3), str(6)
memory usage: 952.8 MB


In [28]:
li_trans_raw.head()

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:08,11,8000ECA90,11,8000ECA90,3195403.00,US Dollar,3195403.00,US Dollar,Reinvestment,0
1,2022/09/01 00:21,3402,80021DAD0,3402,80021DAD0,1858.96,US Dollar,1858.96,US Dollar,Reinvestment,0
2,2022/09/01 00:00,11,8000ECA90,1120,8006AA910,592571.00,US Dollar,592571.00,US Dollar,Cheque,0
3,2022/09/01 00:16,3814,8006AD080,3814,8006AD080,12.32,US Dollar,12.32,US Dollar,Reinvestment,0
4,2022/09/01 00:00,20,8006AD530,20,8006AD530,2941.56,US Dollar,2941.56,US Dollar,Reinvestment,0


## 8. 계좌 데이터 확이

In [32]:
hi_accounts_raw = raw_samples['HI']['accounts']

print("===== HI-Small 계좌 데이터")
print("현재 불러온 크기: ", hi_accounts_raw.shape)

hi_accounts_raw.info()

===== HI-Small 계좌 데이터
현재 불러온 크기:  (518581, 5)
<class 'pandas.DataFrame'>
RangeIndex: 518581 entries, 0 to 518580
Data columns (total 5 columns):
 #   Column          Non-Null Count   Dtype
---  ------          --------------   -----
 0   Bank Name       518581 non-null  str  
 1   Bank ID         518581 non-null  int64
 2   Account Number  518581 non-null  str  
 3   Entity ID       518581 non-null  str  
 4   Entity Name     518581 non-null  str  
dtypes: int64(1), str(4)
memory usage: 47.3 MB


In [33]:
li_accounts_raw = raw_samples['LI']['accounts']

print("===== LI-Small 계좌 데이터")
print("현재 불러온 크기: ", li_accounts_raw.shape)

li_accounts_raw.info()

===== LI-Small 계좌 데이터
현재 불러온 크기:  (712688, 5)
<class 'pandas.DataFrame'>
RangeIndex: 712688 entries, 0 to 712687
Data columns (total 5 columns):
 #   Column          Non-Null Count   Dtype
---  ------          --------------   -----
 0   Bank Name       712688 non-null  str  
 1   Bank ID         712688 non-null  int64
 2   Account Number  712688 non-null  str  
 3   Entity ID       712688 non-null  str  
 4   Entity Name     712688 non-null  str  
dtypes: int64(1), str(4)
memory usage: 65.4 MB


## 9. 컬럼별 원본 상태 확인

In [36]:
def inspect_raw_dataframe(df, dataset_name):
    """
    DataFrame의 원본 상태를 확인하는 함수.

    확인하는 내용
    -------------
    1. 행과 열의 개수
    2. Pandas가 자동으로 판단한 dtype
    3. 결측값 개수와 비율
    4. 현재 샘플 안에서의 고유값 개수
    5. 각 컬럼의 실제 값 예시

    주의
    ----
    이 함수는 DataFrame을 변경하지 않는다.
    """

    print("\n" + "=" * 80)
    print(dataset_name)
    print("=" * 80)

    print("행과 열:", df.shape)

    print("\n[DataFrame info]")

    df.info()

    # 컴럼별 검사 결과를 저장할 리스트 
    profile_rows = []

    for column in df.columns:
        series = df[column]

        # 결측값을 제외한 값
        non_null = series.dropna()

        # 중복되지 않은 실제 값 예시 최대 3개
        examples = (
            non_null
            .astype(str)
            .drop_duplicates()
            .head(3)
            .tolist()
        )

        profile_rows.append({
            "column": column,

            # Pandas가 자동으로 판단한 자료형
            "pandas_dtype": str(series.dtype),

            # 결측값 개수
            "missing_count": int(
                series.isna().sum()
            ),

            # 결측값 비율
            "missing_ratio_percent": round(
                series.isna().mean() * 100,
                4,
            ),

            # 현재 읽은 샘플 안에서의 고유값 수
            "unique_count_in_sample": int(
                series.nunique(dropna=True)
            ),

            # 실제 값 예시
            "examples": examples,
            
        })

    # 검사 결과를 표로 변환
    profile = pd.DataFrame(profile_rows)

    print("\n[컬럼별 프로파일]")
    display(profile)

    print("\n[앞의 10행]")
    display(df.head(10))

    return profile

In [40]:
transactions_profiles = {}

for ratio in RATIOS:
    transactions_profiles[ratio] = inspect_raw_dataframe(
        raw_samples[ratio]["transactions"],
        f"{ratio}-{SIZE} 거래 데이터"
    )



HI-Small 거래 데이터
행과 열: (5078345, 11)

[DataFrame info]
<class 'pandas.DataFrame'>
RangeIndex: 5078345 entries, 0 to 5078344
Data columns (total 11 columns):
 #   Column              Dtype  
---  ------              -----  
 0   Timestamp           str    
 1   From Bank           int64  
 2   Account             str    
 3   To Bank             int64  
 4   Account.1           str    
 5   Amount Received     float64
 6   Receiving Currency  str    
 7   Amount Paid         float64
 8   Payment Currency    str    
 9   Payment Format      str    
 10  Is Laundering       int64  
dtypes: float64(2), int64(3), str(6)
memory usage: 699.6 MB

[컬럼별 프로파일]


,column,pandas_dtype,missing_count,missing_ratio_percent,unique_count_in_sample,examples
0,Timestamp,str,0,0.0,15018,"[2022/09/01 00:20, 2022/09/01 00:00, 2022/09/01 00:02]"
1,From Bank,int64,0,0.0,30470,"[10, 3208, 3209]"
2,Account,str,0,0.0,496995,"[8000EBD30, 8000F4580, 8000F4670]"
3,To Bank,int64,0,0.0,15811,"[10, 1, 3209]"
4,Account.1,str,0,0.0,420636,"[8000EBD30, 8000F5340, 8000F4670]"
5,Amount Received,float64,0,0.0,915161,"[3697.34, 0.01, 14675.57]"
6,Receiving Currency,str,0,0.0,15,"[US Dollar, Bitcoin, Euro]"
7,Amount Paid,float64,0,0.0,923873,"[3697.34, 0.01, 14675.57]"
8,Payment Currency,str,0,0.0,15,"[US Dollar, Bitcoin, Euro]"
9,Payment Format,str,0,0.0,7,"[Reinvestment, Cheque, Credit Card]"



[앞의 10행]


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:20,10,8000EBD30,10,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,0
1,2022/09/01 00:20,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0
2,2022/09/01 00:00,3209,8000F4670,3209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0
3,2022/09/01 00:02,12,8000F5030,12,8000F5030,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,0
4,2022/09/01 00:06,10,8000F5200,10,8000F5200,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,0
5,2022/09/01 00:03,1,8000F5AD0,1,8000F5AD0,6162.44,US Dollar,6162.44,US Dollar,Reinvestment,0
6,2022/09/01 00:08,1,8000EBAC0,1,8000EBAC0,14.26,US Dollar,14.26,US Dollar,Reinvestment,0
7,2022/09/01 00:16,1,8000EC1E0,1,8000EC1E0,11.86,US Dollar,11.86,US Dollar,Reinvestment,0
8,2022/09/01 00:26,12,8000EC280,2439,8017BF800,7.66,US Dollar,7.66,US Dollar,Credit Card,0
9,2022/09/01 00:21,1,8000EDEC0,211050,80AEF5310,383.71,US Dollar,383.71,US Dollar,Credit Card,0



LI-Small 거래 데이터
행과 열: (6924049, 11)

[DataFrame info]
<class 'pandas.DataFrame'>
RangeIndex: 6924049 entries, 0 to 6924048
Data columns (total 11 columns):
 #   Column              Dtype  
---  ------              -----  
 0   Timestamp           str    
 1   From Bank           int64  
 2   Account             str    
 3   To Bank             int64  
 4   Account.1           str    
 5   Amount Received     float64
 6   Receiving Currency  str    
 7   Amount Paid         float64
 8   Payment Currency    str    
 9   Payment Format      str    
 10  Is Laundering       int64  
dtypes: float64(2), int64(3), str(6)
memory usage: 952.8 MB

[컬럼별 프로파일]


,column,pandas_dtype,missing_count,missing_ratio_percent,unique_count_in_sample,examples
0,Timestamp,str,0,0.0,14533,"[2022/09/01 00:08, 2022/09/01 00:21, 2022/09/01 00:00]"
1,From Bank,int64,0,0.0,41814,"[11, 3402, 3814]"
2,Account,str,0,0.0,681281,"[8000ECA90, 80021DAD0, 8006AD080]"
3,To Bank,int64,0,0.0,21588,"[11, 3402, 1120]"
4,Account.1,str,0,0.0,576176,"[8000ECA90, 80021DAD0, 8006AA910]"
5,Amount Received,float64,0,0.0,1194921,"[3195403.0, 1858.96, 592571.0]"
6,Receiving Currency,str,0,0.0,15,"[US Dollar, Euro, Bitcoin]"
7,Amount Paid,float64,0,0.0,1204309,"[3195403.0, 1858.96, 592571.0]"
8,Payment Currency,str,0,0.0,15,"[US Dollar, Euro, Bitcoin]"
9,Payment Format,str,0,0.0,7,"[Reinvestment, Cheque, ACH]"



[앞의 10행]


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:08,11,8000ECA90,11,8000ECA90,3195403.00,US Dollar,3195403.00,US Dollar,Reinvestment,0
1,2022/09/01 00:21,3402,80021DAD0,3402,80021DAD0,1858.96,US Dollar,1858.96,US Dollar,Reinvestment,0
2,2022/09/01 00:00,11,8000ECA90,1120,8006AA910,592571.00,US Dollar,592571.00,US Dollar,Cheque,0
3,2022/09/01 00:16,3814,8006AD080,3814,8006AD080,12.32,US Dollar,12.32,US Dollar,Reinvestment,0
4,2022/09/01 00:00,20,8006AD530,20,8006AD530,2941.56,US Dollar,2941.56,US Dollar,Reinvestment,0
5,2022/09/01 00:24,12,8006ADD30,12,8006ADD30,6473.62,US Dollar,6473.62,US Dollar,Reinvestment,0
6,2022/09/01 00:17,11,800059120,1217,8006AD4E0,60562.00,US Dollar,60562.00,US Dollar,ACH,0
7,2022/09/01 00:07,11,8000ECA90,11,8000ECA90,22.97,US Dollar,22.97,US Dollar,Reinvestment,0
8,2022/09/01 00:28,1120,8006AA910,243166,81470DCF0,43.53,US Dollar,43.53,US Dollar,Credit Card,0
9,2022/09/01 00:22,1217,8006AD4E0,1217,8006AD4E0,5.04,US Dollar,5.04,US Dollar,Reinvestment,0


In [41]:
account_profiles = {}

for ratio in RATIOS:
    account_profiles[ratio] = inspect_raw_dataframe(
        raw_samples[ratio]["accounts"],
        f"{ratio}-{SIZE} 계좌 데이터 "
    )


HI-Small 계좌 데이터 
행과 열: (518581, 5)

[DataFrame info]
<class 'pandas.DataFrame'>
RangeIndex: 518581 entries, 0 to 518580
Data columns (total 5 columns):
 #   Column          Non-Null Count   Dtype
---  ------          --------------   -----
 0   Bank Name       518581 non-null  str  
 1   Bank ID         518581 non-null  int64
 2   Account Number  518581 non-null  str  
 3   Entity ID       518581 non-null  str  
 4   Entity Name     518581 non-null  str  
dtypes: int64(1), str(4)
memory usage: 47.3 MB

[컬럼별 프로파일]


,column,pandas_dtype,missing_count,missing_ratio_percent,unique_count_in_sample,examples
0,Bank Name,str,0,0.0,20053,"[Portugal Bank #4507, Canada Bank #27, UK Bank #33]"
1,Bank ID,int64,0,0.0,30470,"[331579, 210, 21884]"
2,Account Number,str,0,0.0,518573,"[80B779D80, 809D86900, 80812BE00]"
3,Entity ID,str,0,0.0,166207,"[80062E240, 800C998A0, 800C47F50]"
4,Entity Name,str,0,0.0,166207,"[Sole Proprietorship #50438, Corporation #33520, Partnership #35397]"



[앞의 10행]


,Bank Name,Bank ID,Account Number,Entity ID,Entity Name
0,Portugal Bank #4507,331579,80B779D80,80062E240,Sole Proprietorship #50438
1,Canada Bank #27,210,809D86900,800C998A0,Corporation #33520
2,UK Bank #33,21884,80812BE00,800C47F50,Partnership #35397
3,Germany Bank #4815,32742,81047F300,80096F0B0,Corporation #48813
4,National Bank of Harrisburg,127390,80BD8CF00,800FB8760,Corporation #889
5,Spain Bank #439,224555,80F269580,80064EB20,Sole Proprietorship #42987
6,Savings Bank of Omaha,32013,80E6E8680,800CCEDE0,Sole Proprietorship #20855
7,Brazil Bank #39,335355,80CDEFD80,800D88F10,Partnership #36822
8,Mexico Bank #16,1132,80B723600,800D3F760,Partnership #36511
9,Russia Bank #39,217824,806B17000,800B44480,Partnership #33830



LI-Small 계좌 데이터 
행과 열: (712688, 5)

[DataFrame info]
<class 'pandas.DataFrame'>
RangeIndex: 712688 entries, 0 to 712687
Data columns (total 5 columns):
 #   Column          Non-Null Count   Dtype
---  ------          --------------   -----
 0   Bank Name       712688 non-null  str  
 1   Bank ID         712688 non-null  int64
 2   Account Number  712688 non-null  str  
 3   Entity ID       712688 non-null  str  
 4   Entity Name     712688 non-null  str  
dtypes: int64(1), str(4)
memory usage: 65.4 MB

[컬럼별 프로파일]


,column,pandas_dtype,missing_count,missing_ratio_percent,unique_count_in_sample,examples
0,Bank Name,str,0,0.0,27652,"[China Bank #2820, France Bank #4585, China Bank #2242]"
1,Bank ID,int64,0,0.0,41815,"[314693, 311253, 39996]"
2,Account Number,str,0,0.0,712684,"[81B86A280, 8187FEA80, 803961E00]"
3,Entity ID,str,0,0.0,224931,"[800D8CCF0, 800B505E0, 800D03F60]"
4,Entity Name,str,0,0.0,224931,"[Corporation #41344, Corporation #54497, Partnership #36904]"



[앞의 10행]


,Bank Name,Bank ID,Account Number,Entity ID,Entity Name
0,China Bank #2820,314693,81B86A280,800D8CCF0,Corporation #41344
1,France Bank #4585,311253,8187FEA80,800B505E0,Corporation #54497
2,China Bank #2242,39996,803961E00,800D03F60,Partnership #36904
3,National Bank of Newport,331440,81B075800,801567C10,Corporation #16224
4,UK Bank #33,135417,80CF87C80,801085E00,Partnership #72930
5,Savings Bank of Harrisburg,270107,81A2F8D80,80145CEC0,Partnership #17131
6,Crytpo Bank #22,170412,819F4D601,8015B10D0,Partnership #953
7,Brazil Bank #19,148894,811EACF00,801208FE0,Corporation #48389
8,Japan Bank #654,322173,808753E80,800E622A0,Corporation #42450
9,Italy Bank #1554,52130,81455D100,800C1C190,Sole Proprietorship #29256


## 10. 범주형 컬럼 값 확인

In [45]:
# 거래 데이터에서 값의 종류를 확인할 컬럼
columns_to_check = [
    "Receiving Currency",
    "Payment Currency",
    "Payment Format",
    "Is Laundering",
]

for ratio in RATIOS:
    df = raw_samples[ratio]["transactions"]

    print("\n" + "=" * 80)
    print(f"{ratio}-SIZE 주요 범주형 컬럼")
    print("=" * 80)

    for column in columns_to_check:
        # 해당 컬럼이 없으면 건너뜀
        if column not in df.columns:
            print("컬럼 없음:", column)
            continue

        print(f"\n[{column}]")

        # 결속값도 포함에 값별 개수 계산
        counts = (
            df[column]
            .value_counts(dropna=False)
            .rename("count")
            .to_frame()
        )

        counts['ratio_percent'] = (
            counts['count']
            / len(df)
            * 100
        ).round(4)

        display(counts.head(30))


HI-SIZE 주요 범주형 컬럼

[Receiving Currency]


,count,ratio_percent
Receiving Currency,,
US Dollar,1879341,37.0070
Euro,1172017,23.0787
Swiss Franc,237884,4.6843
Yuan,206551,4.0673
Shekel,194988,3.8396
Rupee,192065,3.7820
UK Pound,181255,3.5692
Ruble,157361,3.0987
Yen,156319,3.0781



[Payment Currency]


,count,ratio_percent
Payment Currency,,
US Dollar,1895172,37.3187
Euro,1168297,23.0055
Swiss Franc,234860,4.6247
Yuan,213752,4.2091
Shekel,192184,3.7844
Rupee,190202,3.7454
UK Pound,180738,3.5590
Yen,155209,3.0563
Ruble,155178,3.0557



[Payment Format]


...

,count,ratio_percent
Payment Currency,,
US Dollar,2553887,36.8843
Euro,1595859,23.0481
Yuan,483603,6.9844
Rupee,340641,4.9197
Bitcoin,309240,4.4662
Saudi Riyal,257948,3.7254
Australian Dollar,211155,3.0496
Yen,210125,3.0347
Brazil Real,199840,2.8862



[Payment Format]


,count,ratio_percent
Payment Format,,
Cheque,2503158,36.1517
Credit Card,1780389,25.7131
ACH,796581,11.5046
Cash,655688,9.4697
Reinvestment,650458,9.3942
Bitcoin,309208,4.4657
Wire,228567,3.3011



[Is Laundering]


,count,ratio_percent
Is Laundering,,
0,6920484,99.9485
1,3565,0.0515


## 11. 수치형 컬럼값 확인

In [48]:
for ratio in RATIOS:
    df = raw_samples[ratio]['transactions']

    print("\n" + "=" * 80)
    print(f"{ratio}-{SIZE} 숫자형 컬럼 요약")
    print("=" * 80)

    numeric_summary = df.describe(
        include="number",
        percentiles=[
            0.01,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99,
        ],
    ).T

    display(numeric_summary)
    


HI-Small 숫자형 컬럼 요약


,count,mean,std,min,1%,25%,50%,75%,95%,99%,max
From Bank,5078345.0,4.573057e+04,8.176562e+04,1.000000,1.000000,119.00,9679.00,28628.00,238190.00,3.332860e+05,3.563030e+05
To Bank,5078345.0,6.574456e+04,8.409299e+04,1.000000,3.000000,4259.00,21568.00,122332.00,238151.00,2.545650e+05,3.562940e+05
Amount Received,5078345.0,5.988726e+06,1.037183e+09,0.000001,0.016808,183.37,1411.01,12346.27,649906.51,1.475123e+07,1.046302e+12
Amount Paid,5078345.0,4.509273e+06,8.697728e+08,0.000001,0.018477,184.48,1414.54,12297.84,623757.22,1.352453e+07,1.046302e+12
Is Laundering,5078345.0,1.019427e-03,3.191219e-02,0.000000,0.000000,0.00,0.00,0.00,0.00,0.000000e+00,1.000000e+00



LI-Small 숫자형 컬럼 요약


,count,mean,std,min,1%,25%,50%,75%,95%,99%,max
From Bank,6924049.0,5.938718e+04,9.051700e+04,0.000000,2.000000,219.00,14195.00,110682.00,255833.000,3.457680e+05,3.769670e+05
To Bank,6924049.0,8.441702e+04,9.064562e+04,0.000000,5.000000,11255.00,29640.00,148040.00,249566.000,2.724090e+05,3.769670e+05
Amount Received,6924049.0,6.324067e+06,2.105371e+09,0.000001,0.009488,174.21,1397.62,12296.33,623594.670,1.336849e+07,3.644854e+12
Amount Paid,6924049.0,4.676036e+06,1.544099e+09,0.000001,0.010000,175.38,1399.44,12226.87,600085.324,1.236483e+07,3.644854e+12
Is Laundering,6924049.0,5.148722e-04,2.268495e-02,0.000000,0.000000,0.00,0.00,0.00,0.000,0.000000e+00,1.000000e+00


## 12. 문자열 원본과 Pandas 자동 추론 결과 비교

In [50]:
text_samples = {}

for ratio in RATIOS:
    # 모든 컬럼을 문자열로 읽음
    text_samples[ratio] = pd.read_csv(
        dataset_paths[ratio]["transactions"],
        nrows=100,
        dtype="string",
        low_memory=False,
    )

In [51]:
id_columns = [
    "From Bank",
    "Account",
    "To Bank",
    "Account.1",
]

for ratio in RATIOS:
    default_df = raw_samples[ratio]["transactions"]
    text_df = text_samples[ratio]

    print("\n" + "=" * 80)
    print(f"{ratio}-{SIZE} ID 컬럼 비교")
    print("=" * 80)

    for column in id_columns:
        if column not in default_df.columns:
            continue

        print(f"\n[{column}]")
        print(
            "Pandas 자동 dtype:",
            default_df[column].dtype,
        )

        comparison = pd.DataFrame({
            # Pandas가 자동으로 추론한 결과
            "pandas_default": (
                default_df[column]
                .head(20)
                .reset_index(drop=True)
            ),

            # 원본을 문자열로 강제해서 읽은 결과
            "original_as_string": (
                text_df[column]
                .head(20)
                .reset_index(drop=True)
            ),
        })

        display(comparison)


HI-Small ID 컬럼 비교

[From Bank]
Pandas 자동 dtype: int64


,pandas_default,original_as_string
0,10,010
1,3208,03208
2,3209,03209
3,12,012
4,10,010
5,1,001
6,1,001
7,1,001
8,12,012
9,1,001



[Account]
Pandas 자동 dtype: str


,pandas_default,original_as_string
0,8000EBD30,8000EBD30
1,8000F4580,8000F4580
2,8000F4670,8000F4670
3,8000F5030,8000F5030
4,8000F5200,8000F5200
5,8000F5AD0,8000F5AD0
6,8000EBAC0,8000EBAC0
7,8000EC1E0,8000EC1E0
8,8000EC280,8000EC280
9,8000EDEC0,8000EDEC0



[To Bank]
Pandas 자동 dtype: int64


...

,pandas_default,original_as_string
0,8000ECA90,8000ECA90
1,80021DAD0,80021DAD0
2,8000ECA90,8000ECA90
3,8006AD080,8006AD080
4,8006AD530,8006AD530
5,8006ADD30,8006ADD30
6,800059120,800059120
7,8000ECA90,8000ECA90
8,8006AA910,8006AA910
9,8006AD4E0,8006AD4E0



[To Bank]
Pandas 자동 dtype: int64


,pandas_default,original_as_string
0,11,011
1,3402,03402
2,1120,001120
3,3814,03814
4,20,020
5,12,012
6,1217,01217
7,11,011
8,243166,0243166
9,1217,01217



[Account.1]
Pandas 자동 dtype: str


,pandas_default,original_as_string
0,8000ECA90,8000ECA90
1,80021DAD0,80021DAD0
2,8006AA910,8006AA910
3,8006AD080,8006AD080
4,8006AD530,8006AD530
5,8006ADD30,8006ADD30
6,8006AD4E0,8006AD4E0
7,8000ECA90,8000ECA90
8,81470DCF0,81470DCF0
9,8006AD4E0,8006AD4E0


## 13. HI 와 LI 패턴 파일 원문 확인

In [52]:
# 각 패턴 파일에서 확인할 최대 줄 수
PATTERN_PREVIEW_LINES = 50

for ratio in RATIOS:
    path = dataset_paths[ratio]["patterns"]

    print("\n" + "=" * 80)
    print(f"{ratio}-{SIZE} 패턴 파일 앞부분")
    print("=" * 80)

    with open(
        path,
        mode="r",
        encoding="utf-8",
        errors="replace",
    ) as file:

        for line_number, line in enumerate(
            file,
            start=1,
        ):
            print(
                f"{line_number:>3}: "
                f"{line.rstrip()}"
            )

            if line_number >= PATTERN_PREVIEW_LINES:
                break


HI-Small 패턴 파일 앞부분
  1: BEGIN LAUNDERING ATTEMPT - FAN-OUT:  Max 16-degree Fan-Out
  2: 2022/09/01 00:06,021174,800737690,012,80011F990,2848.96,Euro,2848.96,Euro,ACH,1
  3: 2022/09/01 04:33,021174,800737690,020,80020C5B0,8630.40,Euro,8630.40,Euro,ACH,1
  4: 2022/09/01 09:14,021174,800737690,020,80006A5E0,35642.49,Yuan,35642.49,Yuan,ACH,1
  5: 2022/09/01 09:56,021174,800737690,00220,8007A5B70,5738987.96,US Dollar,5738987.96,US Dollar,ACH,1
  6: 2022/09/01 11:28,021174,800737690,001244,80093C0D0,7254.53,US Dollar,7254.53,US Dollar,ACH,1
  7: 2022/09/01 13:13,021174,800737690,00513,80078E200,6990.87,US Dollar,6990.87,US Dollar,ACH,1
  8: 2022/09/01 14:11,021174,800737690,020,80066B990,12536.92,Euro,12536.92,Euro,ACH,1
  9: 2022/09/02 15:40,021174,800737690,00410,8002CC310,3511.82,Euro,3511.82,Euro,ACH,1
 10: 2022/09/02 21:23,021174,800737690,01292,8004030A0,16135.09,US Dollar,16135.09,US Dollar,ACH,1
 11: 2022/09/02 23:10,021174,800737690,01601,800578800,12183.28,US Dollar,12183.28,US Do

## 14. 패턴 이름별 attempt 개수만 확인

In [53]:
from collections import Counter

pattern_attempt_counts = {}

for ratio in RATIOS:
    path = dataset_paths[ratio]["patterns"]

    # 패턴별 등장 횟수를 저장
    counter = Counter()

    with open(
        path,
        mode="r",
        encoding="utf-8",
        errors="replace",
    ) as file:

        for line in file:
            line = line.strip()

            prefix = "BEGIN LAUNDERING ATTEMPT - "

            # 패턴 시작 행만 선택
            if not line.startswith(prefix):
                continue

            # 접두사 이후의 패턴 정보 추출
            pattern_meta = line[len(prefix):]

            # "CYCLE: Max 12 hops"처럼 부가 정보가 있다면
            # 콜론 앞부분만 패턴 이름으로 사용
            pattern_type = (
                pattern_meta
                .split(":", 1)[0]
                .strip()
                .upper()
            )

            counter[pattern_type] += 1

    pattern_attempt_counts[ratio] = counter

In [54]:
for ratio in RATIOS:
    print(f"\n===== {ratio}-{SIZE} 패턴별 attempt 수 =====")

    result = (
        pd.Series(
            pattern_attempt_counts[ratio],
            name="attempt_count",
        )
        .sort_index()
        .to_frame()
    )

    display(result)


===== HI-Small 패턴별 attempt 수 =====


,attempt_count
BIPARTITE,49
CYCLE,54
FAN-IN,40
FAN-OUT,48
GATHER-SCATTER,51
RANDOM,41
SCATTER-GATHER,44
STACK,43



===== LI-Small 패턴별 attempt 수 =====


,attempt_count
BIPARTITE,16
CYCLE,12
FAN-IN,12
FAN-OUT,19
GATHER-SCATTER,12
RANDOM,15
SCATTER-GATHER,13
STACK,18


### 결과정리
- Timestamp는 현재 문자열이므로 이후 datetime64[ns]로 변환해야 함
- From Bank, To Bank, Bank ID는 현재 int64임 - 식별자는 string으로 관리하는 것이 적절해 보임
- HI의 Account Number는 518,581행 중 518,573개로 계좌번호가 8개 중복
- LI의 Account Number는 712,688행 중 712,684개로 계좌번호가 4개 중복
- Entity ID와 Entity Name의 유니크 수는 두 데이터셋 모두 동일, Entity ID가 하나의 Entity Name에 대응하는 것으로 보이지만, 별도 교차표로 확인해야함

- Pandas가 자동 추론한 컬럼들 수정해야함
| 컬럼 | CSV 저장 형태 | Pandas 자동 추론 타입 | 의미상 권장 타입 |
|---|---|---|---|
| `Timestamp` | 텍스트 | `str` | `datetime64[ns]` |
| `From Bank` | 텍스트 | `int64` | `string` |
| `Account` | 텍스트 | `str` | `string` |
| `Amount Received` | 텍스트 | `float64` | `float64` |
| `Is Laundering` | 텍스트 | `int64` | `Int8` 또는 `bool` |